## --Read Bronze Table --

In [0]:
from src.constants import *
from src.common_functions import *
from src.validations import *

import src.common_functions as cf

print(dir(cf))


In [0]:
laborposition_df = spark.table(LABOR_BRONZE_TABLE)

display(laborposition_df)

In [0]:
print(laborposition_df.count())

--Data Validation--

In [0]:
laborposition_df = trim_columns(laborposition_df)

In [0]:
laborposition_df = replace_blank_with_null(laborposition_df)

In [0]:
from pyspark.sql.functions import col

laborposition_df = laborposition_df.withColumn(
    "Labor_Position_Code",
    col("Labor_Position_Code").cast("int")
)

--Remove Duplicates--

In [0]:
laborposition_df = laborposition_df.dropDuplicates(["Labor_Position_Code"])

print("Rows:", laborposition_df.count())

--Null Validation--

In [0]:
from pyspark.sql import functions as F

print("-- Null Validation --")

null_validation_df = laborposition_df.filter(
    F.col("Labor_Position_Code").isNull()
)

print(
    "Rows with NULL Labor_Position_Code:",
    null_validation_df.count()
)

display(null_validation_df)

In [0]:
laborposition_df.printSchema()

In [0]:
display(
    laborposition_df.filter(
        F.col("Labor_Position_Code") == 9510
    )
)

In [0]:
print("Silver Labor Position row count:", laborposition_df.count())

## Add watermark import

In [0]:
from src.watermark import get_watermark, update_watermark
from pyspark.sql import functions as F

print("Watermark framework imported")

## Read the Labor Position watermark

## Create the incremental DataFrame

In [0]:
# -- Create incremental DataFrame --

from pyspark.sql import functions as F

# Get the latest stored watermark directly from the watermark table
pipeline_name = "Silver Labor Position"
source_name = "Labor_Position.xlsx"

current_labor_watermark = get_watermark(
    pipeline_name,
    source_name
)

print(
    "Current Labor Position watermark:",
    current_labor_watermark
)

# Convert ingestion_timestamp to timestamp
laborposition_candidate_df = (
    laborposition_df
    .withColumn(
        "_ingestion_ts",
        F.to_timestamp("ingestion_timestamp")
    )
)

# Show Bronze maximum ingestion timestamp
source_max_timestamp = (
    laborposition_candidate_df
    .agg(
        F.max("_ingestion_ts")
        .alias("max_ingestion_timestamp")
    )
    .collect()[0]["max_ingestion_timestamp"]
)

print(
    "Bronze Labor Position max timestamp:",
    source_max_timestamp
)

# ---------------------------------------------------------
# Step 1: Apply watermark
# ---------------------------------------------------------

if current_labor_watermark is None:

    laborposition_candidate_df = (
        laborposition_candidate_df
    )

else:

    laborposition_candidate_df = (
        laborposition_candidate_df
        .filter(
            F.col("_ingestion_ts") >
            F.lit(current_labor_watermark)
        )
    )

print(
    "Records newer than watermark:",
    laborposition_candidate_df.count()
)

# ---------------------------------------------------------
# Step 2: Remove Labor Position codes already in Silver
# ---------------------------------------------------------

existing_labor_codes_df = (
    spark.table(LABOR_SILVER_TABLE)
    .select("Labor_Position_Code")
    .distinct()
)

laborposition_incremental_df = (
    laborposition_candidate_df
    .join(
        existing_labor_codes_df,
        on="Labor_Position_Code",
        how="left_anti"
    )
    .drop("_ingestion_ts")
)

incremental_count = laborposition_incremental_df.count()

print(
    "Incremental Labor Position records:",
    incremental_count
)

--Write to silver --

In [0]:
# -- Incremental Silver Load --

incremental_count = laborposition_incremental_df.count()

if incremental_count == 0:

    print("No new Labor Position records to load.")
    print("Silver Labor Position table remains unchanged.")

else:

    laborposition_incremental_df.write \
        .mode("append") \
        .saveAsTable(LABOR_SILVER_TABLE)

    print(
        f"Incremental Labor Position records loaded: {incremental_count}"
    )

    # Update watermark only after successful Silver load
    new_labor_watermark = (
        laborposition_incremental_df
        .withColumn(
            "_ingestion_ts",
            F.to_timestamp("ingestion_timestamp")
        )
        .agg(
            F.max("_ingestion_ts")
            .alias("max_ingestion_timestamp")
        )
        .collect()[0]["max_ingestion_timestamp"]
    )

    update_watermark(
        pipeline_name,
        source_name,
        new_labor_watermark
    )

    print(
        "Labor Position watermark updated to:",
        new_labor_watermark
    )

--Validate Silver--

In [0]:
silver_check_df = spark.table(LABOR_SILVER_TABLE)

print("Silver Labor Position count:", silver_check_df.count())

print(
    "Distinct Labor Position codes:",
    silver_check_df
        .select("Labor_Position_Code")
        .distinct()
        .count()
)

print(
    "Duplicate Labor Position codes:",
    silver_check_df
        .groupBy("Labor_Position_Code")
        .count()
        .filter(F.col("count") > 1)
        .count()
)

In [0]:
silver_check_df = spark.table(
    "databricks_project1.silver.labor_position"
)

print("Silver table count:", silver_check_df.count())

display(
    silver_check_df.filter(
        F.col("Labor_Position_Code") == 9510
    )
)

In [0]:
laborposition_df.printSchema()

In [0]:
display(
    laborposition_df
    .filter(
        F.col("Labor_Position_Code") == 9510
    )
)

In [0]:
#validate
silver_check_df = spark.table(LABOR_SILVER_TABLE)

print(
    "Silver Labor Position count:",
    silver_check_df.count()
)

print(
    "Distinct Labor Position codes:",
    silver_check_df
        .select("Labor_Position_Code")
        .distinct()
        .count()
)

print(
    "Duplicate Labor Position codes:",
    silver_check_df
        .groupBy("Labor_Position_Code")
        .count()
        .filter(F.col("count") > 1)
        .count()
)